# PDF Structure Evaluation and Tokenization Analysis

## Objectives

1. **Identify Document Sections**: Detect headers, references, methodology, and other key sections within PDF documents
2. **Extract Structured Content**: Parse and organize text from PDF pages
3. **Analyze Text Layout**: Study positioning, formatting, and spatial relationships of text elements
4. **Develop Robust Tokenization**: Create reliable methods for breaking down PDF content into meaningful units
5. **Compare Detection Approaches**: Evaluate text-based vs. layout-based detection methods

## Rationale

PDFs present unique challenges for text extraction:
- Text positioning varies across documents and page layouts
- Section headers may lack consistent formatting
- Layout information is critical for understanding document structure
- Traditional text-only approaches miss visual/positional context

This analysis uses multiple detection strategies:
- **Text Heuristics**: Keyword matching with confidence scoring
- **Layout Analysis**: Position-based detection via pdfplumber
- **Hybrid Approach**: Combines text and layout signals

## Current Analysis

- **Document**: Academic research paper (EJIM journal)
- **Total pages**: 26
- **Extracted words (current page)**: 255
- **Y-position groups (layout clusters)**: 11   

In [ ]:
# pick one of these three
pdf_path='../data/0e20b252-374a-8055-3ce5-67225751e3ce.pdf'
pdf_path='../data/5f3b02b4-e497-39bf-2339-4c3c0a55968e.pdf'
pdf_path='../data/17af2c40-3c32-fc5f-7937-f73141ea979a.pdf'

from pypdf import PdfReader

reader = PdfReader(pdf_path)
text = ""
for page in reader.pages:
    text += page.extract_text() + "\n"


In [36]:
print(text[0:1000])

A survey on incumbent digital
transformation: a paradoxical
perspective and research agenda
Tiziano Volpentesta and Esli Spahiu
Luiss University, Rome, Italy, and
Pietro De Giovanni
SDA Bocconi School of Management, Milan, Italy
Abstract
Purpose – Digital transformation (DT) is a major challenge for incumbent organisations, as research on this
phenomenon has revealed a high failure rate. Given this consideration, this paper reviews the literature on DT
in incumbent organisations to identify the main themes and research directions to be undertaken.
Design/methodology/approach – The authors adopt a systematic literature review (SLR) and
computational literature review (CLR) employing a machine learning algorithm for topic modelling (LDA) to
surface the themes discussed in 103 peer-reviewed studies published between 2010 and 2022 in a
multidisciplinary article sample.
Findings – The authors identify and discuss the five main themes emerging from the studies, offering the
state-of-the-art 

In [37]:
text = ""
found_references = False

# Common reference section markers (case-insensitive)
reference_markers = [
    "references",
    "bibliography",
    "works cited",
    "citations",
    "cited works",
]

for page in reader.pages:
    page_text = page.extract_text()

    # If we haven't found references yet, check this page
    if not found_references:
        page_lower = page_text.lower()
        for marker in reference_markers:
            if marker in page_lower:
                # Find the position and extract from the marker onwards
                marker_pos = page_lower.find(marker)
                text += page_text[marker_pos:]
                print(f"Found '{marker}' section, extracting references from here")
                found_references = True
                break
    else:
        # After finding references, include all subsequent pages
        text += page_text + "\n"

# If no reference section found, return empty (better than sending whole PDF)
if not found_references:
    print("No reference section found in PDF")

Found 'bibliography' section, extracting references from here


In [38]:
print(len(text))
print(len(reader.pages))

68048
24


In [39]:
# Analyze reader.pages directly to find References section
# without using the pre-processed text variable

reference_markers = [
    "references",
    "bibliography",
    "works cited",
    "citations",
    "cited works",
]

for page_num, page in enumerate(reader.pages):
    page_text = page.extract_text()
    lines = page_text.split('\n')
    
    for line_num, line in enumerate(lines):
        line_lower = line.strip().lower()
        
        # Check each marker
        for marker in reference_markers:
            # More precise matching: marker should be the primary content on this line
            if (line_lower == marker or 
                (len(line_lower) < 50 and 
                 line_lower.split()[0] == marker if line_lower.split() else False)):
                
                print(f"✓ Found '{marker}' on page {page_num+1}, line {line_num+1}")
                print(f"  Full line: '{line.strip()}'")
                print(f"  Context (next line): '{lines[line_num+1].strip() if line_num+1 < len(lines) else 'N/A'}'")
                print()

✓ Found 'references' on page 16, line 30
  Full line: 'References'
  Context (next line): 'Agarwal, R., Gao, G.G., DesRoches, C. and Jha, A.K. (2010), “Research commentary— the digital'



In [40]:
# Sophisticated document structure analysis using multiple heuristics
# Looks for actual document sections, not just keyword matches

def analyze_document_structure(reader):
    """
    Analyze PDF structure using:
    1. Line isolation - is it on its own line?
    2. Text formatting - ALL CAPS, title case, short length
    3. Positional context - appears at section boundaries
    4. Contextual clues - what comes before/after?
    """
    
    header_keywords = {
        'references', 'bibliography', 'works cited', 'citations',
        'abstract', 'introduction', 'methodology', 'results', 'keywords'
        'discussion', 'conclusion', 'appendix', 'acknowledgments'
    }
    
    findings = []
    
    for page_num, page in enumerate(reader.pages):
        page_text = page.extract_text()
        lines = page_text.split('\n')
        
        for line_num, line in enumerate(lines):
            stripped = line.strip()
            
            # Skip empty or very short lines
            if not stripped or len(stripped) < 2:
                continue
            
            # Heuristic 1: Is this line in a header-like format?
            word_count = len(stripped.split())
            is_short = len(stripped) < 50
            is_formatted = stripped.isupper() or stripped.istitle()
            
            # Heuristic 2: Does it match known headers?
            matches_keyword = any(keyword in stripped.lower() for keyword in header_keywords)
            first_word = stripped.split()[0].lower() if stripped.split() else ""
            exact_keyword_match = first_word in header_keywords
            
            # Heuristic 3: Is it isolated (line before/after are empty or different)?
            prev_empty = line_num == 0 or not lines[line_num-1].strip()
            next_empty = line_num == len(lines)-1 or not lines[line_num+1].strip()
            
            # Heuristic 4: Is it NOT preceded by typical inline markers?
            prev_line = lines[line_num-1].strip() if line_num > 0 else ""
            not_inline = not any(marker in prev_line.lower() for marker in ['see', 'as shown', 'refer to', 'discussed in'])
            
            # Calculate confidence score
            score = 0
            reasons = []
            
            if exact_keyword_match:
                score += 40
                reasons.append("exact keyword")
            elif matches_keyword:
                score += 20
                reasons.append("contains keyword")
            
            if is_short and is_formatted:
                score += 30
                reasons.append("short + formatted")
            elif is_short:
                score += 15
                reasons.append("short")
            
            if (prev_empty or next_empty) and word_count <= 3:
                score += 20
                reasons.append("isolated")
            
            if not_inline and score > 30:
                score += 10
                reasons.append("not inline")
            
            # Report if score suggests a section header
            if score >= 50:
                findings.append({
                    'page': page_num + 1,
                    'line': line_num + 1,
                    'text': stripped,
                    'score': score,
                    'reasons': reasons,
                    'context_before': lines[line_num-1].strip()[:50] if line_num > 0 else 'START',
                    'context_after': lines[line_num+1].strip()[:50] if line_num < len(lines)-1 else 'END'
                })
    
    return findings

# Run analysis
print("SOPHISTICATED DOCUMENT STRUCTURE ANALYSIS")
print("=" * 70)

findings = analyze_document_structure(reader)

for finding in findings:
    print(f"\nPage {finding['page']}, Line {finding['line']} | Score: {finding['score']}")
    print(f"  Text: '{finding['text']}'")
    print(f"  Confidence factors: {', '.join(finding['reasons'])}")
    print(f"  Before: '{finding['context_before']}'")
    print(f"  After:  '{finding['context_after']}'")

print(f"\n{'-' * 70}")
print(f"Found {len(findings)} likely section headers")

SOPHISTICATED DOCUMENT STRUCTURE ANALYSIS

Page 1, Line 8 | Score: 80
  Text: 'Abstract'
  Confidence factors: exact keyword, short + formatted, not inline
  Before: 'SDA Bocconi School of Management, Milan, Italy'
  After:  'Purpose – Digital transformation (DT) is a major c'

Page 1, Line 63 | Score: 60
  Text: 'DOI 10.1108/EJIM-01-2023-0081'
  Confidence factors: short + formatted, isolated, not inline
  Before: '1460-1060'
  After:  'END'

Page 2, Line 6 | Score: 60
  Text: '1. Introduction'
  Confidence factors: contains keyword, short + formatted, not inline
  Before: 'Paper typeLiterature review'
  After:  'Digital transformation (DT) is a complex, interdis'

Page 2, Line 24 | Score: 50
  Text: 'results due to internal silos that hampered the change processes (Lanzolla et al., 2021).'
  Confidence factors: exact keyword, not inline
  Before: 'using case studies, such as the case of General El'
  After:  'Moreover, evidence suggests that the DT of incumbe'

Page 5, Line 25 | Scor

In [41]:
# Analyze PDF structure using pdfplumber for superior document understanding
# pdfplumber extracts text with bounding boxes and layout information

try:
    import pdfplumber
    
    print("PDFPLUMBER - ADVANCED DOCUMENT STRUCTURE ANALYSIS")
    print("=" * 70)
    
    with pdfplumber.open(pdf_path) as pdf:
        print(f"Total pages: {len(pdf.pages)}\n")
        
        # Analyze first few pages for structure
        for page_num, page in enumerate(pdf.pages[:3]):  # First 3 pages
            print(f"\n--- PAGE {page_num + 1} ---")
            print(f"Dimensions: {page.width} x {page.height}")
            
            # Extract text with position information
            text_objects = page.extract_words()
            
            if text_objects:
                print(f"Words found: {len(text_objects)}")
                
                # Look for section headers by analyzing Y-position clusters
                # Headers are typically isolated, larger, or have more spacing
                y_positions = {}
                for obj in text_objects:
                    y = round(obj['top'], 0)  # Round to group similar heights
                    if y not in y_positions:
                        y_positions[y] = []
                    y_positions[y].append(obj['text'])
                
                # Find lines that look like headers (few words, isolated Y-position)
                print("\nPotential headers (by Y-position isolation):")
                for y in sorted(y_positions.keys())[:10]:  # Show top candidates
                    line_text = ' '.join(y_positions[y])
                    word_count = len(y_positions[y])
                    
                    # Header indicators: few words, short line, isolated Y-position
                    if word_count <= 3 and len(line_text) < 50:
                        print(f"  Y:{y:6.0f} | {word_count} words | '{line_text}'")
            
            # Extract tables if any (shows structure)
            tables = page.extract_tables()
            if tables:
                print(f"Tables found: {len(tables)}")
            
            # Get lines and rectangles (visual structure)
            lines = page.lines
            rects = page.rects
            print(f"Visual elements - Lines: {len(lines)}, Rectangles: {len(rects)}")
    
    print("\n" + "=" * 70)
    print("pdfplumber advantages:")
    print("✓ Precise character coordinates (x, y positions)")
    print("✓ Detect visual structure (lines, rectangles, tables)")
    print("✓ Extract tables with structure preservation")
    print("✓ Better for analyzing layout-based hierarchy")
    print("✓ Can identify isolated text regions")
    
except ImportError:
    print("pdfplumber not installed. Install with: pip install pdfplumber")
    print("\nWhy use pdfplumber?")
    print("- Extracts text WITH bounding box coordinates")
    print("- Detects tables, lines, visual elements")
    print("- Better for understanding document layout")
    print("- Can identify headers by isolation and positioning")
    print("- More reliable for structure detection than text heuristics")

PDFPLUMBER - ADVANCED DOCUMENT STRUCTURE ANALYSIS
Total pages: 24


--- PAGE 1 ---
Dimensions: 493.228 x 680.315
Words found: 144

Potential headers (by Y-position isolation):
  Y:    46 | 1 words | 'https://www.emerald.com/insight/1460-1060.htm'
  Y:    81 | 1 words | 'EJIM'
  Y:    93 | 1 words | '26,7'
  Y:   101 | 3 words | 'transformation: a paradoxical'
  Y:   148 | 1 words | '478'
  Y:   163 | 1 words | 'LuissUniversity,Rome,Italy,and'
Visual elements - Lines: 0, Rectangles: 1

--- PAGE 2 ---
Dimensions: 493.228 x 680.315
Words found: 253

Potential headers (by Y-position isolation):
  Y:    91 | 1 words | 'incumbents’'
  Y:   105 | 1 words | 'digital'
  Y:   111 | 1 words | 'Computationalliteraturereview'
  Y:   117 | 1 words | 'transformation'
  Y:   121 | 1 words | 'PapertypeLiteraturereview'
  Y:   148 | 1 words | '479'
  Y:   154 | 1 words | '1.Introduction'
Visual elements - Lines: 0, Rectangles: 2

--- PAGE 3 ---
Dimensions: 493.228 x 680.315
Words found: 255

Potential h